# 04D1B_v2 — LANDFIRE Independent Provenance Audit

**Build ID: `04D1B_V2_LANDFIRE_PROVENANCE_AUDIT`**

04D1A V5 established internal consistency between the official ArcGIS ImageServer's:

- `exportImage`
- `getSamples`

for LF2016R, LF2022, and LF2023.

This notebook asks a different question:

> **Does the LF2023 ArcGIS ImageServer agree with an independently served LANDFIRE raster source?**

The audit has two arms.

---

## Arm A — WCS spot check

Official LANDFIRE CONUS LF2023 WCS:

`https://edcintl.cr.usgs.gov/geoserver/landfire_wcs/conus_2023/wcs`

The WCS is served by GeoServer rather than the ArcGIS REST ImageServer used in 04D1A.

For **LF2023 FBFM40**, this notebook:

1. discovers the exact WCS coverage identifier from `GetCapabilities`;
2. reads an existing 11×11 V5 ArcGIS raw TIFF;
3. requests the same footprint from WCS in EPSG:5070;
4. opens the returned GeoTIFF;
5. samples WCS at the exact cell centers of the V5 TIFF;
6. compares all valid values exactly.

This is a cheap independent delivery-stack check.

---

## Arm B — Full Extent artifact

Official LANDFIRE Full Extent package:

`LF2023_FBFM40_CONUS.zip`

Expected:

- published size: 3.27 GB
- published MD5:
  `6f8404d0c77453e53f2185634bee97d8`

The full download is a much stronger provenance anchor because its integrity is tied to a publisher-supplied checksum.

**The 3.27 GB download is OFF by default.**

After Arm A passes, set:

```python
RUN_FULL_EXTENT_DOWNLOAD = True
```

and rerun the Full Extent sections.

The notebook will:

1. download/resume the ZIP;
2. verify MD5;
3. inspect ZIP member names/sizes;
4. identify the FBFM40 TIFF;
5. optionally extract it only if sufficient disk is available;
6. compare the same V5 11×11 source window against the Full Extent TIFF.

---

## Scientific interpretation

A pass establishes consistency across:

\[
\text{ArcGIS ImageServer}
\leftrightarrow
\text{GeoServer WCS}
\leftrightarrow
\text{publisher-checksummed Full Extent artifact}
\]

This validates **distribution, packaging, grid registration, and raw-value integrity**.

It does not make the different public delivery paths independent of the underlying LANDFIRE production process.


## V2 methodological hardening

V1 already demonstrated 121/121 value agreement between ArcGIS and WCS.

V2 adds checks that cannot be hidden by coordinate lookup:

1. **exact transform identity** between ArcGIS and WCS 11×11 TIFFs;
2. **direct NumPy array equality** after transform identity is established;
3. a **13×13 WCS enclosing-window test** whose inner 11×11 must reproduce the original 11×11 exactly;
4. explicit WCS 1.0.0 `DescribeCoverage` inspection for output formats/interpolation;
5. Full Extent ZIP **CRC-32 verification** via `ZipFile.testzip()`;
6. full-raster dimensions/origin derived from the live LF2023 ImageServer rather than copied from external prose;
7. Full Extent comparison uses direct source-grid array extraction after geometry identity is verified.

These changes do not alter the already accepted 04D1A V5 raw extraction semantics.


## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04D1B_V2_LANDFIRE_PROVENANCE_AUDIT"

from pathlib import Path
import sys, subprocess, importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

V5_RASTER_ROOT = (
    PROJECT_ROOT / "data" / "raw" /
    "landfire" / "pilot_exports_v5"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_provenance_audit_v2"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "landfire_provenance_audit_v2"
)

RAW_ROOT = (
    PROJECT_ROOT / "data" / "raw" /
    "landfire" / "full_extent_lf2023"
)

for p in [OUTPUT_ROOT, PROCESSED_ROOT, RAW_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

assert V5_RASTER_ROOT.exists(), V5_RASTER_ROOT

# ---------- Official WCS ----------
LF2023_WCS_URL = (
    "https://edcintl.cr.usgs.gov/"
    "geoserver/landfire_wcs/conus_2023/wcs"
)


LF2023_FBFM40_IMAGESERVER_URL = (
    "https://lfps.usgs.gov/arcgis/rest/services/"
    "Landfire_LF2023/LF2023_FBFM40_CONUS/ImageServer"
)

# ---------- Full Extent ----------
FULL_EXTENT_URL = (
    "https://landfire.gov/data-downloads/"
    "CONUS_LF2023/LF2023_FBFM40_CONUS.zip"
)

FULL_EXTENT_ZIP = (
    RAW_ROOT / "LF2023_FBFM40_CONUS.zip"
)

EXPECTED_MD5 = "6f8404d0c77453e53f2185634bee97d8"

# 3.27 GB public-page value is a rounded display value.
EXPECTED_PUBLIC_SIZE_TEXT = "3.27 GB"

# Safe defaults:
RUN_FULL_EXTENT_DOWNLOAD = True
RUN_FULL_EXTENT_EXTRACTION = True

REQUEST_TIMEOUT_S = 120
DOWNLOAD_CHUNK_BYTES = 8 * 1024 * 1024

PRODUCT = "FBFM40"
VERSION_KEY = "LF2023"

NODATA_VALUE = -9999
EXPECTED_WKID = 5070
EXPECTED_PIXEL_M = 30.0

FBFM40_CODES = {
    91, 92, 93, 98, 99,
    *range(101, 110),
    *range(121, 125),
    *range(141, 150),
    *range(161, 166),
    *range(181, 190),
    *range(201, 205),
}

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("Full Extent download enabled:", RUN_FULL_EXTENT_DOWNLOAD)

## 1. Imports

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "requests": "requests",
    "rasterio": "rasterio",
    "lxml": "lxml",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package]
        )

from datetime import datetime, timezone
import hashlib
import json
import math
import os
import shutil
import time
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
import rasterio

from rasterio.crs import CRS
from rasterio.transform import rowcol
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 340)

session = requests.Session()
session.headers.update({
    "User-Agent": "wildfire-fireline-effectiveness/04D1B"
})

## 2. Locate an existing LF2023 FBFM40 V5 raw TIFF

04D1A V5 wrote SHA-1-safe filenames, so the notebook discovers the file rather than assuming a particular unit ID.

In [ ]:
v5_candidates = sorted(
    V5_RASTER_ROOT.glob("LF2023_FBFM40_*.tif")
)

if not v5_candidates:
    raise RuntimeError(
        f"No LF2023 FBFM40 V5 TIFF found under {V5_RASTER_ROOT}"
    )

print("Candidate V5 LF2023 FBFM40 TIFFs:", len(v5_candidates))

V5_TIFF = v5_candidates[0]

print("Using:")
print(V5_TIFF)

with rasterio.open(V5_TIFF) as src:
    v5_array = src.read(1)

    v5_meta = {
        "width": src.width,
        "height": src.height,
        "count": src.count,
        "dtype": src.dtypes[0],
        "crs": str(src.crs),
        "nodata": src.nodata,
        "pixel_size_x": abs(src.transform.a),
        "pixel_size_y": abs(src.transform.e),
        "bounds_left": src.bounds.left,
        "bounds_bottom": src.bounds.bottom,
        "bounds_right": src.bounds.right,
        "bounds_top": src.bounds.top,
        "transform": tuple(src.transform)[:6],
    }

display(pd.DataFrame([v5_meta]))

assert v5_meta["width"] == 11
assert v5_meta["height"] == 11
assert v5_meta["count"] == 1
assert str(v5_meta["dtype"]).lower() == "int16"
assert CRS.from_user_input(v5_meta["crs"]) == CRS.from_epsg(5070)
assert abs(v5_meta["pixel_size_x"] - 30) < 1e-9
assert abs(v5_meta["pixel_size_y"] - 30) < 1e-9
assert float(v5_meta["nodata"]) == -9999.0

print("V5 SOURCE WINDOW: PASS")

## 3. WCS GetCapabilities

In [ ]:
def get_wcs_capabilities(version="1.0.0"):
    response = session.get(
        LF2023_WCS_URL,
        params={
            "service": "WCS",
            "version": version,
            "request": "GetCapabilities",
        },
        timeout=REQUEST_TIMEOUT_S,
    )
    response.raise_for_status()

    text = response.text

    # Ensure this is XML rather than an HTML/proxy error.
    root = ET.fromstring(text)

    return text, root


wcs_xml, wcs_root = get_wcs_capabilities("1.0.0")

print("WCS GetCapabilities bytes:", len(wcs_xml.encode("utf-8")))
print("Root tag:", wcs_root.tag)

## 4. Discover the exact FBFM40 coverage identifier

In [ ]:
def local_tag(tag):
    return tag.split("}")[-1].lower()


def extract_wcs_coverages(root):
    records = []

    for node in root.iter():
        if local_tag(node.tag) not in {
            "coverageofferingbrief",
            "coveragesummary",
        }:
            continue

        name = None
        label = None
        description = None

        for child in node.iter():
            tag = local_tag(child.tag)
            text = (child.text or "").strip()

            if not text:
                continue

            if tag in {"name", "identifier", "coverageid"} and name is None:
                name = text
            elif tag in {"label", "title"} and label is None:
                label = text
            elif tag in {"description", "abstract"} and description is None:
                description = text

        if name:
            records.append({
                "coverage_id": name,
                "label": label,
                "description": description,
            })

    return pd.DataFrame(records).drop_duplicates()


coverages = extract_wcs_coverages(wcs_root)

print("Coverage records:", len(coverages))

if coverages.empty:
    raise RuntimeError("No WCS coverage identifiers parsed.")

display(coverages.head(100))

def coverage_score(row):
    text = " ".join(
        str(row.get(k) or "")
        for k in ["coverage_id", "label", "description"]
    ).lower()

    score = 0

    for token, points in [
        ("fbfm40", 20),
        ("f40", 15),
        ("2023", 10),
        ("240", 10),
        ("conus", 5),
    ]:
        if token in text:
            score += points

    return score


coverages["score"] = coverages.apply(
    coverage_score,
    axis=1,
)

ranked = coverages.sort_values(
    ["score", "coverage_id"],
    ascending=[False, True],
)

display(ranked.head(20))

best = ranked.iloc[0]

if best["score"] <= 0:
    raise RuntimeError(
        "Could not identify LF2023 FBFM40 WCS coverage."
    )

WCS_COVERAGE_ID = best["coverage_id"]

print("Selected WCS coverage:")
print(WCS_COVERAGE_ID)


## 4B. DescribeCoverage semantics

Before requesting pixels, inspect the exact LF2023 FBFM40 coverage description.

This records:

- advertised output formats;
- advertised interpolation methods;
- coverage CRS/envelope metadata.

The production request remains pinned to **WCS 1.0.0**.


In [ ]:

describe_response = session.get(
    LF2023_WCS_URL,
    params={
        "service": "WCS",
        "version": "1.0.0",
        "request": "DescribeCoverage",
        "coverage": WCS_COVERAGE_ID,
    },
    timeout=REQUEST_TIMEOUT_S,
)
describe_response.raise_for_status()

describe_root = ET.fromstring(describe_response.text)

describe_text = describe_response.text.lower()

advertised_formats = sorted({
    (node.text or "").strip()
    for node in describe_root.iter()
    if local_tag(node.tag) in {"format", "formats"}
    and (node.text or "").strip()
})

advertised_interpolations = sorted({
    (node.text or "").strip()
    for node in describe_root.iter()
    if "interpolation" in local_tag(node.tag)
    and (node.text or "").strip()
})

print("Advertised formats:")
print(advertised_formats)

print()
print("Advertised interpolation strings:")
print(advertised_interpolations)

DESCRIBE_COVERAGE_PATH = (
    OUTPUT_ROOT / "LF2023_FBFM40_WCS_DESCRIBE_COVERAGE.xml"
)
DESCRIBE_COVERAGE_PATH.write_text(
    describe_response.text,
    encoding="utf-8",
)

# Keep GeoTIFF because V1 already demonstrated that this server accepts it.
WCS_FORMAT = "GeoTIFF"

# Use nearest only if an advertised interpolation token can be identified.
nearest_candidates = [
    value
    for value in advertised_interpolations
    if "nearest" in value.lower()
]

WCS_INTERPOLATION = (
    nearest_candidates[0]
    if nearest_candidates
    else None
)

print()
print("Selected WCS format:", WCS_FORMAT)
print("Selected WCS interpolation:", WCS_INTERPOLATION)


## 5. WCS GetCoverage using the exact V5 footprint

WCS 1.0.0 is used because its `BBOX + WIDTH + HEIGHT` request is explicit and easy to align to the existing 11×11 V5 source window.

No reprojection is requested:

- source/request CRS = EPSG:5070
- width = 11
- height = 11
- footprint = exact V5 TIFF bounds

In [ ]:
WCS_TIFF = (
    RAW_ROOT / "LF2023_FBFM40_WCS_11x11.tif"
)

bounds = (
    v5_meta["bounds_left"],
    v5_meta["bounds_bottom"],
    v5_meta["bounds_right"],
    v5_meta["bounds_top"],
)

wcs_params = {
    "service": "WCS",
    "version": "1.0.0",
    "request": "GetCoverage",
    "coverage": WCS_COVERAGE_ID,
    "crs": "EPSG:5070",
    "response_crs": "EPSG:5070",
    "bbox": ",".join(str(v) for v in bounds),
    "width": 11,
    "height": 11,
    "format": WCS_FORMAT,
}

if WCS_INTERPOLATION:
    wcs_params["interpolation"] = WCS_INTERPOLATION

response = session.get(
    LF2023_WCS_URL,
    params=wcs_params,
    timeout=REQUEST_TIMEOUT_S,
)

if not response.ok:
    print("WCS status:", response.status_code)
    print(response.text[:4000])

response.raise_for_status()

content_type = response.headers.get("Content-Type", "")

if (
    "xml" in content_type.lower()
    or response.content[:5].lstrip().startswith(b"<?xml")
):
    print(response.text[:4000])
    raise RuntimeError(
        "WCS GetCoverage returned XML rather than GeoTIFF."
    )

WCS_TIFF.write_bytes(response.content)

print("Saved:", WCS_TIFF)
print("Bytes:", WCS_TIFF.stat().st_size)
print("Content-Type:", content_type)

## 6. Inspect WCS TIFF

In [ ]:
with rasterio.open(WCS_TIFF) as src:
    wcs_array = src.read(1)

    wcs_meta = {
        "width": src.width,
        "height": src.height,
        "count": src.count,
        "dtype": src.dtypes[0],
        "crs": str(src.crs),
        "nodata": src.nodata,
        "pixel_size_x": abs(src.transform.a),
        "pixel_size_y": abs(src.transform.e),
        "bounds_left": src.bounds.left,
        "bounds_bottom": src.bounds.bottom,
        "bounds_right": src.bounds.right,
        "bounds_top": src.bounds.top,
        "transform": tuple(src.transform)[:6],
    }

display(pd.DataFrame([wcs_meta]))

assert wcs_meta["width"] == 11
assert wcs_meta["height"] == 11
assert wcs_meta["count"] == 1

assert CRS.from_user_input(
    wcs_meta["crs"]
) == CRS.from_epsg(5070)

assert abs(
    wcs_meta["pixel_size_x"] - 30.0
) < 1e-6

assert abs(
    wcs_meta["pixel_size_y"] - 30.0
) < 1e-6

print("WCS GRID METADATA: PASS")


## 6B. Exact grid identity and direct-array equality

This is the primary V2 WCS test.

Coordinate sampling is not used to establish equality here.

Required:

- CRS identical;
- width/height identical;
- affine transform coefficients identical to numerical precision expected from GeoTIFF serialization;
- NoData masks identical;
- direct raw arrays identical.

Only after these pass is coordinate sampling retained as a diagnostic.


In [ ]:

with rasterio.open(V5_TIFF) as a, rasterio.open(WCS_TIFF) as b:
    arc_array = a.read(1)
    wcs_array_direct = b.read(1)

    arc_transform = np.asarray(
        tuple(a.transform)[:6],
        dtype=float,
    )
    wcs_transform = np.asarray(
        tuple(b.transform)[:6],
        dtype=float,
    )

    transform_diff = np.abs(
        arc_transform - wcs_transform
    )

    TRANSFORM_EXACT_PASS = bool(
        np.all(
            transform_diff <= 1e-9
        )
    )

    CRS_IDENTITY_PASS = bool(
        a.crs == b.crs
        == CRS.from_epsg(5070)
    )

    SHAPE_IDENTITY_PASS = (
        arc_array.shape
        == wcs_array_direct.shape
        == (11, 11)
    )

    arc_nodata_mask = (
        ~np.isfinite(arc_array)
        | (arc_array == -9999)
    )

    wcs_nodata_mask_direct = (
        ~np.isfinite(wcs_array_direct)
        | (wcs_array_direct == -9999)
    )

    NODATA_MASK_DIRECT_PASS = bool(
        np.array_equal(
            arc_nodata_mask,
            wcs_nodata_mask_direct,
        )
    )

    DIRECT_ARRAY_PASS = bool(
        np.array_equal(
            arc_array,
            wcs_array_direct,
        )
    )

grid_identity = pd.DataFrame([
    {
        "crs_identity": CRS_IDENTITY_PASS,
        "shape_identity": SHAPE_IDENTITY_PASS,
        "transform_identity_1e-9": TRANSFORM_EXACT_PASS,
        "max_transform_abs_diff": float(transform_diff.max()),
        "nodata_mask_identity": NODATA_MASK_DIRECT_PASS,
        "direct_array_equal": DIRECT_ARRAY_PASS,
    }
])

display(grid_identity)

print(
    "WCS EXACT GRID + ARRAY:",
    "PASS"
    if all([
        CRS_IDENTITY_PASS,
        SHAPE_IDENTITY_PASS,
        TRANSFORM_EXACT_PASS,
        NODATA_MASK_DIRECT_PASS,
        DIRECT_ARRAY_PASS,
    ])
    else "REVIEW REQUIRED",
)



## 6C. 13×13 enclosing-window invariance

Request a WCS window expanded by exactly one native 30 m cell on each side.

The returned 13×13 raster must:

- remain on the same grid;
- have 30 m pixels;
- contain the original 11×11 WCS array exactly as `[1:-1, 1:-1]`.

This detects request-extent-dependent resampling or a bbox convention error.


In [ ]:

WCS_TIFF_13 = (
    RAW_ROOT / "LF2023_FBFM40_WCS_13x13.tif"
)

expanded_bounds = (
    v5_meta["bounds_left"] - 30,
    v5_meta["bounds_bottom"] - 30,
    v5_meta["bounds_right"] + 30,
    v5_meta["bounds_top"] + 30,
)

params_13 = {
    "service": "WCS",
    "version": "1.0.0",
    "request": "GetCoverage",
    "coverage": WCS_COVERAGE_ID,
    "crs": "EPSG:5070",
    "response_crs": "EPSG:5070",
    "bbox": ",".join(
        str(v)
        for v in expanded_bounds
    ),
    "width": 13,
    "height": 13,
    "format": WCS_FORMAT,
}

if WCS_INTERPOLATION:
    params_13["interpolation"] = (
        WCS_INTERPOLATION
    )

r13 = session.get(
    LF2023_WCS_URL,
    params=params_13,
    timeout=REQUEST_TIMEOUT_S,
)
r13.raise_for_status()

if (
    "xml" in r13.headers.get(
        "Content-Type",
        "",
    ).lower()
    or r13.content[:5].lstrip().startswith(b"<?xml")
):
    print(r13.text[:4000])
    raise RuntimeError(
        "13×13 WCS request returned XML."
    )

WCS_TIFF_13.write_bytes(
    r13.content
)

with rasterio.open(WCS_TIFF) as s11, rasterio.open(WCS_TIFF_13) as s13:
    a11 = s11.read(1)
    a13 = s13.read(1)

    WCS_13_SHAPE_PASS = (
        a13.shape == (13, 13)
    )

    WCS_13_PIXEL_PASS = bool(
        abs(s13.transform.a - 30.0) <= 1e-9
        and abs(abs(s13.transform.e) - 30.0) <= 1e-9
    )

    WCS_13_INNER_PASS = bool(
        np.array_equal(
            a13[1:-1, 1:-1],
            a11,
        )
    )

containment_summary = pd.DataFrame([
    {
        "shape_13x13": WCS_13_SHAPE_PASS,
        "native_30m": WCS_13_PIXEL_PASS,
        "inner_11x11_equals_original": WCS_13_INNER_PASS,
    }
])

display(containment_summary)

print(
    "WCS 13x13 CONTAINMENT:",
    "PASS"
    if all([
        WCS_13_SHAPE_PASS,
        WCS_13_PIXEL_PASS,
        WCS_13_INNER_PASS,
    ])
    else "REVIEW REQUIRED",
)


## 7. Compare WCS against the V5 ArcGIS source window

The comparison is location-based rather than relying on array orientation.

Every V5 TIFF cell center is sampled from the returned WCS TIFF using its own geotransform.

In [ ]:
comparison_rows = []

with rasterio.open(V5_TIFF) as v5src, rasterio.open(WCS_TIFF) as wcssrc:
    for row in range(v5src.height):
        for col in range(v5src.width):
            x, y = v5src.xy(
                row,
                col,
                offset="center",
            )

            v5_value = float(
                v5src.read(
                    1,
                    window=((row, row+1), (col, col+1)),
                )[0, 0]
            )

            wrow, wcol = wcssrc.index(
                x,
                y,
            )

            inside = (
                0 <= wrow < wcssrc.height
                and 0 <= wcol < wcssrc.width
            )

            wcs_value = (
                float(
                    wcssrc.read(
                        1,
                        window=((wrow, wrow+1), (wcol, wcol+1)),
                    )[0, 0]
                )
                if inside
                else np.nan
            )

            v5_nodata = (
                not np.isfinite(v5_value)
                or v5_value == -9999
            )

            wcs_nodata = (
                not np.isfinite(wcs_value)
                or wcs_value == -9999
            )

            valid_comparable = (
                not v5_nodata
                and not wcs_nodata
            )

            exact = (
                bool(v5_value == wcs_value)
                if valid_comparable
                else False
            )

            domain_v5 = (
                v5_nodata
                or int(v5_value) in FBFM40_CODES
            )

            domain_wcs = (
                wcs_nodata
                or int(wcs_value) in FBFM40_CODES
            )

            comparison_rows.append({
                "row": row,
                "col": col,
                "x": x,
                "y": y,
                "v5_value": v5_value,
                "wcs_value": wcs_value,
                "v5_nodata": v5_nodata,
                "wcs_nodata": wcs_nodata,
                "valid_comparable": valid_comparable,
                "exact_valid_match": exact,
                "v5_domain_ok": domain_v5,
                "wcs_domain_ok": domain_wcs,
            })


wcs_compare = pd.DataFrame(
    comparison_rows
)

valid = wcs_compare[
    wcs_compare["valid_comparable"]
]

wcs_summary = pd.DataFrame([
    {
        "pixels": len(wcs_compare),
        "valid_comparable_pixels": len(valid),
        "exact_valid_matches": int(valid["exact_valid_match"].sum()),
        "exact_valid_match_pct": (
            float(valid["exact_valid_match"].mean() * 100)
            if len(valid)
            else np.nan
        ),
        "v5_nodata_pixels": int(wcs_compare["v5_nodata"].sum()),
        "wcs_nodata_pixels": int(wcs_compare["wcs_nodata"].sum()),
        "nodata_mask_exact": bool(
            np.array_equal(
                wcs_compare["v5_nodata"].to_numpy(bool),
                wcs_compare["wcs_nodata"].to_numpy(bool),
            )
        ),
        "all_v5_domain_ok": bool(wcs_compare["v5_domain_ok"].all()),
        "all_wcs_domain_ok": bool(wcs_compare["wcs_domain_ok"].all()),
    }
])

display(wcs_summary)

wcs_compare.to_parquet(
    PROCESSED_ROOT / "lf2023_fbfm40_wcs_vs_v5_pixels.parquet",
    index=False,
)

wcs_summary.to_csv(
    OUTPUT_ROOT / "LF2023_FBFM40_WCS_VS_V5_SUMMARY.csv",
    index=False,
)

WCS_PASS = bool(
    len(valid) >= 80
    and valid["exact_valid_match"].all()
    and wcs_compare["v5_domain_ok"].all()
    and wcs_compare["wcs_domain_ok"].all()
    and CRS_IDENTITY_PASS
    and SHAPE_IDENTITY_PASS
    and TRANSFORM_EXACT_PASS
    and NODATA_MASK_DIRECT_PASS
    and DIRECT_ARRAY_PASS
    and WCS_13_SHAPE_PASS
    and WCS_13_PIXEL_PASS
    and WCS_13_INNER_PASS
)

print(
    "WCS PROVENANCE SPOT CHECK:",
    "PASS" if WCS_PASS else "REVIEW REQUIRED",
)


## 7B. Derive expected full-raster geometry from the live LF2023 ImageServer

Do not hard-code the claimed CONUS dimensions.

Use the live service's own:

- extent;
- pixel size;
- pixel type;

to derive the expected full-grid geometry used later for the publisher artifact.


In [ ]:

live_meta_response = session.get(
    LF2023_FBFM40_IMAGESERVER_URL,
    params={"f": "pjson"},
    timeout=REQUEST_TIMEOUT_S,
)
live_meta_response.raise_for_status()

live_meta = live_meta_response.json()

if "error" in live_meta:
    raise RuntimeError(
        live_meta["error"]
    )

extent = live_meta["extent"]

LIVE_PIXEL_X = abs(
    float(live_meta["pixelSizeX"])
)
LIVE_PIXEL_Y = abs(
    float(live_meta["pixelSizeY"])
)

LIVE_XMIN = float(
    extent["xmin"]
)
LIVE_YMIN = float(
    extent["ymin"]
)
LIVE_XMAX = float(
    extent["xmax"]
)
LIVE_YMAX = float(
    extent["ymax"]
)

EXPECTED_FULL_WIDTH = int(
    round(
        (LIVE_XMAX - LIVE_XMIN)
        / LIVE_PIXEL_X
    )
)

EXPECTED_FULL_HEIGHT = int(
    round(
        (LIVE_YMAX - LIVE_YMIN)
        / LIVE_PIXEL_Y
    )
)

live_grid = pd.DataFrame([
    {
        "xmin": LIVE_XMIN,
        "ymin": LIVE_YMIN,
        "xmax": LIVE_XMAX,
        "ymax": LIVE_YMAX,
        "pixel_x": LIVE_PIXEL_X,
        "pixel_y": LIVE_PIXEL_Y,
        "expected_width": EXPECTED_FULL_WIDTH,
        "expected_height": EXPECTED_FULL_HEIGHT,
        "pixel_type": live_meta.get("pixelType"),
        "uncompressed_size_reported": live_meta.get("uncompressedSize"),
    }
])

display(live_grid)

assert abs(LIVE_PIXEL_X - 30.0) <= 1e-9
assert abs(LIVE_PIXEL_Y - 30.0) <= 1e-9

print(
    "Expected full raster:",
    EXPECTED_FULL_WIDTH,
    "×",
    EXPECTED_FULL_HEIGHT,
)


## 8. Full Extent package — preflight only

The official LANDFIRE page publishes:

- product: LF2023 FBFM40 CONUS
- display size: 3.27 GB
- MD5: `6f8404d0c77453e53f2185634bee97d8`

No large download occurs unless `RUN_FULL_EXTENT_DOWNLOAD=True`.

In [ ]:
disk = shutil.disk_usage(RAW_ROOT)

disk_summary = pd.DataFrame([
    {
        "path": str(RAW_ROOT),
        "total_gb": disk.total / (1024**3),
        "used_gb": disk.used / (1024**3),
        "free_gb": disk.free / (1024**3),
        "download_enabled": RUN_FULL_EXTENT_DOWNLOAD,
        "extraction_enabled": RUN_FULL_EXTENT_EXTRACTION,
        "expected_public_zip_size": EXPECTED_PUBLIC_SIZE_TEXT,
        "expected_md5": EXPECTED_MD5,
    }
])

display(disk_summary)

print("Full Extent URL:")
print(FULL_EXTENT_URL)

## 9. Resumable Full Extent download

In [ ]:
def download_with_resume(
    url,
    output_path,
):
    existing = (
        output_path.stat().st_size
        if output_path.exists()
        else 0
    )

    headers = {}

    if existing > 0:
        headers["Range"] = f"bytes={existing}-"

    with session.get(
        url,
        headers=headers,
        stream=True,
        timeout=REQUEST_TIMEOUT_S,
    ) as response:

        if existing > 0 and response.status_code == 206:
            mode = "ab"
        elif response.status_code == 200:
            # Server ignored Range; start clean.
            mode = "wb"
            existing = 0
        else:
            response.raise_for_status()
            mode = "wb"

        total_remaining = pd.to_numeric(
            response.headers.get("Content-Length"),
            errors="coerce",
        )

        if np.isfinite(total_remaining):
            total_remaining = int(total_remaining)
        else:
            total_remaining = None

        with open(output_path, mode) as f:
            with tqdm(
                total=total_remaining,
                unit="B",
                unit_scale=True,
                desc="LF2023 FBFM40 Full Extent",
            ) as bar:
                for chunk in response.iter_content(
                    chunk_size=DOWNLOAD_CHUNK_BYTES
                ):
                    if not chunk:
                        continue

                    f.write(chunk)
                    bar.update(len(chunk))

    return output_path


if RUN_FULL_EXTENT_DOWNLOAD:
    download_with_resume(
        FULL_EXTENT_URL,
        FULL_EXTENT_ZIP,
    )

    print("Downloaded:")
    print(FULL_EXTENT_ZIP)
    print(
        "ZIP size GB:",
        FULL_EXTENT_ZIP.stat().st_size / (1024**3),
    )
else:
    print(
        "SKIPPED. Set RUN_FULL_EXTENT_DOWNLOAD=True "
        "after the WCS check if you want the definitive archive audit."
    )

## 10. MD5 verification

In [ ]:
def file_md5(path, chunk_bytes=16*1024*1024):
    digest = hashlib.md5()

    total = path.stat().st_size

    with open(path, "rb") as f:
        with tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc="MD5",
        ) as bar:
            while True:
                chunk = f.read(chunk_bytes)

                if not chunk:
                    break

                digest.update(chunk)
                bar.update(len(chunk))

    return digest.hexdigest()


FULL_EXTENT_MD5 = None
FULL_EXTENT_MD5_PASS = None

if FULL_EXTENT_ZIP.exists():
    FULL_EXTENT_MD5 = file_md5(
        FULL_EXTENT_ZIP
    )

    FULL_EXTENT_MD5_PASS = (
        FULL_EXTENT_MD5.lower()
        == EXPECTED_MD5.lower()
    )

    print("Observed MD5:", FULL_EXTENT_MD5)
    print("Expected MD5:", EXPECTED_MD5)
    print("MD5 PASS:", FULL_EXTENT_MD5_PASS)

    if not FULL_EXTENT_MD5_PASS:
        raise RuntimeError(
            "Full Extent ZIP MD5 does not match LANDFIRE-published checksum."
        )
else:
    print("Full Extent ZIP not present; MD5 stage skipped.")

## 11. Inspect ZIP contents and extraction requirement

In [ ]:
zip_inventory = pd.DataFrame()
FULL_EXTENT_TIF_MEMBER = None
ZIP_CRC_PASS = None

if FULL_EXTENT_ZIP.exists():
    with zipfile.ZipFile(FULL_EXTENT_ZIP, "r") as zf:
        # CRC-32 integrity of every archive member.
        bad_member = zf.testzip()

        ZIP_CRC_PASS = (
            bad_member is None
        )

        print(
            "ZIP CRC-32:",
            "PASS"
            if ZIP_CRC_PASS
            else f"FAIL at {bad_member}"
        )

        if not ZIP_CRC_PASS:
            raise RuntimeError(
                f"ZIP CRC failure at member: {bad_member}"
            )

        rows = []

        for info in zf.infolist():
            rows.append({
                "member": info.filename,
                "compressed_gb": info.compress_size / (1024**3),
                "uncompressed_gb": info.file_size / (1024**3),
                "compression_type": info.compress_type,
            })

        zip_inventory = pd.DataFrame(rows)

    display(
        zip_inventory.sort_values(
            "uncompressed_gb",
            ascending=False,
        ).head(50)
    )

    tif_candidates = zip_inventory[
        zip_inventory["member"]
        .str.lower()
        .str.endswith((".tif", ".tiff"))
    ].copy()

    print("GeoTIFF candidates:", len(tif_candidates))
    display(tif_candidates)

    if len(tif_candidates) == 1:
        FULL_EXTENT_TIF_MEMBER = tif_candidates.iloc[0]["member"]

    elif len(tif_candidates) > 1:
        scored = tif_candidates.copy()
        scored["score"] = (
            scored["member"]
            .str.lower()
            .map(
                lambda x:
                10 * ("f40" in x)
                + 10 * ("fbfm40" in x)
                + 5 * ("240" in x)
            )
        )

        FULL_EXTENT_TIF_MEMBER = (
            scored.sort_values(
                ["score", "uncompressed_gb"],
                ascending=[False, False],
            )
            .iloc[0]["member"]
        )

    print("Selected TIFF member:")
    print(FULL_EXTENT_TIF_MEMBER)

    zip_inventory.to_csv(
        OUTPUT_ROOT / "LF2023_FBFM40_FULL_EXTENT_ZIP_INVENTORY.csv",
        index=False,
    )
else:
    print("ZIP not downloaded; inventory skipped.")

## 12. Optional extraction of the official TIFF

This stage is separately gated because the uncompressed raster may require tens of GB.

The notebook checks free disk space against the ZIP member's uncompressed size plus a 5 GB safety margin.

In [ ]:
FULL_EXTENT_TIF_PATH = None

if (
    RUN_FULL_EXTENT_EXTRACTION
    and FULL_EXTENT_ZIP.exists()
    and FULL_EXTENT_TIF_MEMBER
):
    with zipfile.ZipFile(FULL_EXTENT_ZIP, "r") as zf:
        info = zf.getinfo(FULL_EXTENT_TIF_MEMBER)

        required = (
            info.file_size
            + 5 * 1024**3
        )

        free = shutil.disk_usage(RAW_ROOT).free

        print(
            "Required incl. margin GB:",
            required / (1024**3),
        )

        print(
            "Free GB:",
            free / (1024**3),
        )

        if free < required:
            raise RuntimeError(
                "Not enough free disk space for safe Full Extent TIFF extraction."
            )

        extracted = Path(
            zf.extract(
                FULL_EXTENT_TIF_MEMBER,
                path=RAW_ROOT,
            )
        )

        FULL_EXTENT_TIF_PATH = extracted

        print("Extracted:")
        print(FULL_EXTENT_TIF_PATH)

elif RUN_FULL_EXTENT_EXTRACTION:
    print(
        "Extraction requested but ZIP/TIFF member is unavailable."
    )
else:
    print(
        "Extraction skipped. Set RUN_FULL_EXTENT_EXTRACTION=True "
        "after the ZIP passes MD5 and you confirm disk capacity."
    )

## 13. Full Extent TIFF metadata and 11×11 comparison

In [ ]:
FULL_EXTENT_COMPARE_PASS = None
full_extent_summary = pd.DataFrame()

if FULL_EXTENT_TIF_PATH and FULL_EXTENT_TIF_PATH.exists():
    with rasterio.open(FULL_EXTENT_TIF_PATH) as src:
        full_meta = {
            "width": src.width,
            "height": src.height,
            "count": src.count,
            "dtype": src.dtypes[0],
            "crs": str(src.crs),
            "nodata": src.nodata,
            "pixel_size_x": abs(src.transform.a),
            "pixel_size_y": abs(src.transform.e),
            "bounds_left": src.bounds.left,
            "bounds_bottom": src.bounds.bottom,
            "bounds_right": src.bounds.right,
            "bounds_top": src.bounds.top,
            "transform": tuple(src.transform)[:6],
        }

        display(pd.DataFrame([full_meta]))

        FULL_GEOMETRY_PASS = bool(
            src.width == EXPECTED_FULL_WIDTH
            and src.height == EXPECTED_FULL_HEIGHT
            and src.count == 1
            and str(src.dtypes[0]).lower() == "int16"
            and src.crs == CRS.from_epsg(5070)
            and abs(src.transform.a - 30.0) <= 1e-9
            and abs(abs(src.transform.e) - 30.0) <= 1e-9
            and abs(src.bounds.left - LIVE_XMIN) <= 1e-9
            and abs(src.bounds.right - LIVE_XMAX) <= 1e-9
            and abs(src.bounds.bottom - LIVE_YMIN) <= 1e-9
            and abs(src.bounds.top - LIVE_YMAX) <= 1e-9
        )

        print(
            "FULL EXTENT GRID GEOMETRY:",
            "PASS"
            if FULL_GEOMETRY_PASS
            else "REVIEW REQUIRED"
        )

        rows = []

        with rasterio.open(V5_TIFF) as v5src:
            for row in range(v5src.height):
                for col in range(v5src.width):
                    x, y = v5src.xy(
                        row,
                        col,
                        offset="center",
                    )

                    v5_value = float(
                        v5src.read(
                            1,
                            window=((row, row+1), (col, col+1)),
                        )[0, 0]
                    )

                    frow, fcol = src.index(
                        x,
                        y,
                    )

                    inside = (
                        0 <= frow < src.height
                        and 0 <= fcol < src.width
                    )

                    full_value = (
                        float(
                            src.read(
                                1,
                                window=((frow, frow+1), (fcol, fcol+1)),
                            )[0, 0]
                        )
                        if inside
                        else np.nan
                    )

                    v5_nodata = (
                        not np.isfinite(v5_value)
                        or v5_value == -9999
                    )

                    full_nodata = (
                        not np.isfinite(full_value)
                        or full_value == -9999
                    )

                    valid = (
                        not v5_nodata
                        and not full_nodata
                    )

                    rows.append({
                        "row": row,
                        "col": col,
                        "x": x,
                        "y": y,
                        "v5_value": v5_value,
                        "full_extent_value": full_value,
                        "valid_comparable": valid,
                        "exact_valid_match": (
                            bool(v5_value == full_value)
                            if valid
                            else False
                        ),
                        "v5_nodata": v5_nodata,
                        "full_extent_nodata": full_nodata,
                    })

    full_compare = pd.DataFrame(rows)

    valid = full_compare[
        full_compare["valid_comparable"]
    ]

    full_extent_summary = pd.DataFrame([
        {
            "pixels": len(full_compare),
            "valid_comparable_pixels": len(valid),
            "exact_valid_matches": int(
                valid["exact_valid_match"].sum()
            ),
            "exact_valid_match_pct": (
                float(
                    valid["exact_valid_match"].mean() * 100
                )
                if len(valid)
                else np.nan
            ),
            "nodata_mask_exact": bool(
                np.array_equal(
                    full_compare["v5_nodata"].to_numpy(bool),
                    full_compare["full_extent_nodata"].to_numpy(bool),
                )
            ),
            "full_width": full_meta["width"],
            "full_height": full_meta["height"],
            "full_dtype": full_meta["dtype"],
            "full_crs": full_meta["crs"],
            "full_pixel_size_x": full_meta["pixel_size_x"],
            "full_pixel_size_y": full_meta["pixel_size_y"],
        }
    ])

    display(full_extent_summary)

    FULL_EXTENT_COMPARE_PASS = bool(
        FULL_GEOMETRY_PASS
        and len(valid) >= 80
        and valid["exact_valid_match"].all()
        and CRS.from_user_input(
            full_meta["crs"]
        ) == CRS.from_epsg(5070)
        and abs(
            full_meta["pixel_size_x"] - 30
        ) < 1e-9
        and abs(
            full_meta["pixel_size_y"] - 30
        ) < 1e-9
    )

    full_compare.to_parquet(
        PROCESSED_ROOT / "lf2023_fbfm40_full_extent_vs_v5_pixels.parquet",
        index=False,
    )

    full_extent_summary.to_csv(
        OUTPUT_ROOT / "LF2023_FBFM40_FULL_EXTENT_VS_V5_SUMMARY.csv",
        index=False,
    )

    print(
        "FULL EXTENT PROVENANCE CHECK:",
        "PASS" if FULL_EXTENT_COMPARE_PASS else "REVIEW REQUIRED",
    )

else:
    print("Full Extent TIFF not extracted; comparison skipped.")

## 14. Final provenance decision

In [ ]:
checks = pd.DataFrame([
    {
        "check": "v5_source_window_valid",
        "pass": True,
        "status": "completed",
    },
    {
        "check": "lf2023_wcs_coverage_discovered",
        "pass": bool(WCS_COVERAGE_ID),
        "status": "completed",
    },
    {
        "check": "wcs_vs_arcgis_exact_valid_pixel_agreement",
        "pass": bool(WCS_PASS),
        "status": "completed",
    },
    {
        "check": "full_extent_zip_crc32",
        "pass": (
            bool(ZIP_CRC_PASS)
            if ZIP_CRC_PASS is not None
            else pd.NA
        ),
        "status": (
            "completed"
            if ZIP_CRC_PASS is not None
            else "not_run"
        ),
    },
    {
        "check": "full_extent_zip_md5",
        "pass": (
            bool(FULL_EXTENT_MD5_PASS)
            if FULL_EXTENT_MD5_PASS is not None
            else pd.NA
        ),
        "status": (
            "completed"
            if FULL_EXTENT_MD5_PASS is not None
            else "not_run"
        ),
    },
    {
        "check": "full_extent_vs_arcgis_exact_valid_pixel_agreement",
        "pass": (
            bool(FULL_EXTENT_COMPARE_PASS)
            if FULL_EXTENT_COMPARE_PASS is not None
            else pd.NA
        ),
        "status": (
            "completed"
            if FULL_EXTENT_COMPARE_PASS is not None
            else "not_run"
        ),
    },
])

display(checks)

if WCS_PASS:
    print("04D1B WCS PROVENANCE ARM: PASS")
else:
    print("04D1B WCS PROVENANCE ARM: REVIEW REQUIRED")

if (
    FULL_EXTENT_MD5_PASS is True
    and ZIP_CRC_PASS is True
    and FULL_EXTENT_COMPARE_PASS is True
):
    print("04D1B FULL EXTENT PROVENANCE ARM: PASS")
    print("04D1B INDEPENDENT PROVENANCE AUDIT: FULL PASS")
else:
    print(
        "Full Extent arm not yet complete. "
        "This does not invalidate the WCS arm."
    )

checks.to_csv(
    OUTPUT_ROOT / "LANDFIRE_PROVENANCE_AUDIT_CHECKS.csv",
    index=False,
)

## 15. Final report

In [ ]:
def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report = f"""# 04D1B LANDFIRE Independent Provenance Audit

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## V5 source TIFF

`{V5_TIFF}`

## WCS

Endpoint:

`{LF2023_WCS_URL}`

Coverage:

`{WCS_COVERAGE_ID}`

### WCS vs ArcGIS V5

{tomd(wcs_summary)}

## Full Extent artifact

URL:

`{FULL_EXTENT_URL}`

Published size:

`{EXPECTED_PUBLIC_SIZE_TEXT}`

Published MD5:

`{EXPECTED_MD5}`

Observed MD5:

`{FULL_EXTENT_MD5}`

### Full Extent comparison

{tomd(full_extent_summary) if len(full_extent_summary) else "Not run."}

## Checks

{tomd(checks)}

## Interpretation

The WCS arm is independent at the server-software/serving-stack level from the ArcGIS REST ImageServer used in 04D1A; both may still share EROS infrastructure and/or underlying raster artifacts.

The Full Extent arm, when completed with publisher-supplied MD5 plus ZIP CRC verification, is the strongest public artifact-level provenance anchor.

These tests validate delivery, packaging, raw-value preservation, and grid registration.

They do not constitute an independent replication of LANDFIRE's underlying fuel-model production methodology.
"""

REPORT_PATH = (
    OUTPUT_ROOT / "LANDFIRE_PROVENANCE_AUDIT_V2_REPORT.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(report)
print()
print("Saved:", REPORT_PATH)

## 16. Output inventory

In [ ]:
print("=" * 100)
print("04D1B OUTPUTS")
print("=" * 100)

for p in sorted(OUTPUT_ROOT.glob("*")):
    if p.is_file():
        print(
            f"{p.name:<85}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print("Raw provenance files:")
for p in sorted(RAW_ROOT.rglob("*")):
    if p.is_file():
        print(
            f"{str(p.relative_to(RAW_ROOT)):<85}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print("DONE:", NOTEBOOK_BUILD_ID)